Quiz 59 — Distribution Simulation  [SOLVED]
Difficulty: Hard

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(220)
n_days = 90

# ── 1. Generate simulation data ───────────────────────────────────────────────
arrivals   = np.random.poisson(lam=25, size=n_days)
group_size = np.random.binomial(n=6, p=0.55, size=n_days)
spend_pp   = np.random.normal(38, 8, n_days).clip(10, 80)
no_shows   = np.random.binomial(n=arrivals, p=0.08)

# ── 2. Actual covers ──────────────────────────────────────────────────────────
actual_covers = np.maximum((arrivals - no_shows) * group_size, 0)

# ── 3. Evening revenue ────────────────────────────────────────────────────────
revenue = actual_covers * spend_pp

# ── 4. Stats ──────────────────────────────────────────────────────────────────
print(f"Mean revenue   : £{revenue.mean():,.2f}")
print(f"Median revenue : £{np.median(revenue):,.2f}")
print(f"Std revenue    : £{revenue.std():,.2f}")
print(f"Mean covers/night: {actual_covers.mean():.1f}")

# ── 5. Top-10 evenings ────────────────────────────────────────────────────────
top10_idx = np.argsort(revenue)[-10:][::-1]
print("\nTop-10 revenue evenings:")
for i in top10_idx:
    print(f"  Day {i+1:3d}: covers={actual_covers[i]:3d}  revenue=£{revenue[i]:,.0f}")

# ── 6. % exceeding £7,000 ─────────────────────────────────────────────────────
pct_7k = (revenue > 7000).mean() * 100
print(f"\nEvenings > £7,000: {pct_7k:.1f}%")

# ── 7. Histogram + normal curve ──────────────────────────────────────────────
mu, sigma = revenue.mean(), revenue.std()
x_range   = np.linspace(revenue.min(), revenue.max(), 200)
normal_pdf = (1/(sigma * np.sqrt(2*np.pi))) * np.exp(-0.5*((x_range-mu)/sigma)**2)

fig, ax = plt.subplots(figsize=(10, 6))
n_hist, bins, _ = ax.hist(revenue, bins=20, density=True,
                           color="steelblue", alpha=0.7, label="Actual revenue")
ax.plot(x_range, normal_pdf, color="red", linewidth=2, label="Normal fit")
ax.axvline(mu, color="navy", linestyle="--", label=f"Mean £{mu:,.0f}")
ax.axvline(7000, color="orange", linestyle="--", label="£7,000 threshold")
ax.set_title("Evening Revenue Distribution (90 Days)")
ax.set_xlabel("Revenue (£)")
ax.set_ylabel("Density")
ax.legend()
plt.tight_layout()
plt.savefig("hard_19_distribution_sim.png", dpi=100)
plt.show()

# ── WHY ───────────────────────────────────────────────────────────────────────
# Combining Poisson (arrivals) + Binomial (group size / no-shows) + Normal (spend)
# creates a realistic compound distribution — real business models work this way.
# density=True normalises the histogram to probability density so it can be
# overlaid with a continuous normal PDF on the same scale.